In [0]:
from pyspark.sql.functions import (
    col, when, lit,
    current_timestamp,
    row_number
)
from pyspark.sql.window import Window

In [0]:
BRONZE_BASE_PATH = "/Volumes/main/default/datalake_retail/bronze"
SILVER_BASE_PATH = "/Volumes/main/default/datalake_retail/silver"

In [0]:
sales_bronze = spark.read.format("delta") \
    .load(f"{BRONZE_BASE_PATH}/sales_transactions")

products_bronze = spark.read.format("delta") \
    .load(f"{BRONZE_BASE_PATH}/products")

stores_bronze = spark.read.format("delta") \
    .load(f"{BRONZE_BASE_PATH}/stores")

In [0]:
#Deduplicate sales by transaction_id
window_spec = Window.partitionBy("transaction_id") \
    .orderBy(col("last_updated_ts").desc())

sales_deduped = (
    sales_bronze
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_prod = Window.partitionBy("product_id") \
    .orderBy(col("last_updated_ts").desc())

products_silver = (
    products_bronze
    .withColumn("row_num", row_number().over(window_prod))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

In [0]:
products_silver = products_silver.fillna({
    "category": "UNKNOWN",
    "brand": "UNKNOWN"
})

In [0]:
products_silver.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/main/default/datalake_retail/silver/products")

In [0]:
window_store = Window.partitionBy("store_id") \
    .orderBy(col("last_updated_ts").desc())

stores_silver = (
    stores_bronze
    .withColumn("row_num", row_number().over(window_store))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

In [0]:
#Data Calibration
sales_calibrated = sales_deduped.withColumn(
    "calculated_total",
    (col("quantity") * col("unit_price")) - col("discount")
)

sales_calibrated = sales_calibrated.withColumn(
    "total_amount",
    when(col("total_amount") != col("calculated_total"),
         col("calculated_total"))
    .otherwise(col("total_amount"))
).drop("calculated_total")

In [0]:
stores_silver = stores_silver.fillna({
    "region": "UNKNOWN",
    "city": "UNKNOWN"
})

In [0]:
stores_silver.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/main/default/datalake_retail/silver/stores")

In [0]:
dbutils.fs.ls("/Volumes/main/default/datalake_retail/silver")

[FileInfo(path='dbfs:/Volumes/main/default/datalake_retail/silver/products/', name='products/', size=0, modificationTime=1766432978923),
 FileInfo(path='dbfs:/Volumes/main/default/datalake_retail/silver/sales_quarantine/', name='sales_quarantine/', size=0, modificationTime=1766432978923),
 FileInfo(path='dbfs:/Volumes/main/default/datalake_retail/silver/sales_transactions/', name='sales_transactions/', size=0, modificationTime=1766432978923),
 FileInfo(path='dbfs:/Volumes/main/default/datalake_retail/silver/stores/', name='stores/', size=0, modificationTime=1766432978923)]

In [0]:
#Referential Validation
valid_products = products_bronze \
    .filter(col("active_flag") == True) \
    .select("product_id")

valid_stores = stores_bronze \
    .filter(col("active_flag") == True) \
    .select("store_id")

In [0]:
#Join for validation
sales_validated = (
    sales_calibrated
    .join(valid_products, "product_id", "left")
    .join(valid_stores, "store_id", "left")
)

In [0]:
sales_with_rejection = sales_validated.withColumn(
    "rejection_reason",
    when(col("quantity") <= 0, "Invalid quantity")
    .when(col("unit_price") <= 0, "Invalid unit price")
    .when(col("product_id").isNull(), "Invalid product")
    .when(col("store_id").isNull(), "Invalid store")
    .otherwise(lit(None))
)

In [0]:
#Split valid vs quarantine
sales_clean = sales_with_rejection.filter(col("rejection_reason").isNull())
sales_quarantine = sales_with_rejection.filter(col("rejection_reason").isNotNull())

In [0]:
from delta.tables import DeltaTable

silver_sales_path = f"{SILVER_BASE_PATH}/sales_transactions"

if DeltaTable.isDeltaTable(spark, silver_sales_path):
    delta_table = DeltaTable.forPath(spark, silver_sales_path)

    delta_table.alias("t").merge(
        sales_clean.alias("s"),
        "t.transaction_id = s.transaction_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
else:
    sales_clean.write.format("delta") \
        .mode("overwrite") \
        .save(silver_sales_path)

In [0]:
sales_quarantine.write.format("delta") \
    .mode("append") \
    .save(f"{SILVER_BASE_PATH}/sales_quarantine")

In [0]:
#Validation
spark.read.format("delta").load(silver_sales_path).count()

7611

In [0]:
spark.read.format("delta").load(f"{SILVER_BASE_PATH}/sales_quarantine").count()

2389

Delta Lake Implementation

In [0]:
%sql
UPDATE delta.`/Volumes/main/default/datalake_retail/silver/sales_transactions`
SET discount = 0
WHERE discount IS NULL


In [0]:
%sql
UPDATE delta.`/Volumes/main/default/datalake_retail/silver/sales_transactions`
SET currency = 'INR'
WHERE currency IS NULL

num_affected_rows
0


In [0]:
%sql
DESCRIBE HISTORY delta.`/Volumes/main/default/datalake_retail/silver/sales_transactions`

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2025-12-22T19:42:26.000Z,72297342688245,mishamanmai2004@gmail.com,UPDATE,"Map(predicate -> [""isnull(currency#14886)""])",null,List(1038742538391076),1222-182653-qr7k1m6j-v2n,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 614, numDeletionVectorsUpdated -> 0, scanTimeMs -> 573, numAddedFiles -> 0, numUpdatedRows -> 0, numAddedBytes -> 0, rewriteTimeMs -> 0)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13
0,2025-12-22T19:14:49.000Z,72297342688245,mishamanmai2004@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(1038742538391076),1222-182653-qr7k1m6j-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 7611, numOutputBytes -> 254775)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13


In [0]:
%sql
SELECT COUNT(*)
FROM delta.`/Volumes/main/default/datalake_retail/silver/sales_transactions`
VERSION AS OF 0

COUNT(*)
7611


In [0]:
%sql
SELECT COUNT(*)
FROM delta.`/Volumes/main/default/datalake_retail/silver/sales_transactions`

COUNT(*)
7611


In [0]:
%sql
RESTORE TABLE delta.`/Volumes/main/default/datalake_retail/silver/sales_transactions`
TO VERSION AS OF 0

table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
254775,1,0,0,0,0


In [0]:
%sql
DESCRIBE HISTORY delta.`/Volumes/main/default/datalake_retail/silver/sales_transactions`

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2025-12-22T19:43:30.000Z,72297342688245,mishamanmai2004@gmail.com,RESTORE,"Map(version -> 0, timestamp -> null)",null,List(1038742538391076),1222-182653-qr7k1m6j-v2n,1,Serializable,false,"Map(numRestoredFiles -> 0, removedFilesSize -> 0, numRemovedFiles -> 0, restoredFilesSize -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numOfFilesAfterRestore -> 1, tableSizeAfterRestore -> 254775)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13
1,2025-12-22T19:42:26.000Z,72297342688245,mishamanmai2004@gmail.com,UPDATE,"Map(predicate -> [""isnull(currency#14886)""])",null,List(1038742538391076),1222-182653-qr7k1m6j-v2n,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 614, numDeletionVectorsUpdated -> 0, scanTimeMs -> 573, numAddedFiles -> 0, numUpdatedRows -> 0, numAddedBytes -> 0, rewriteTimeMs -> 0)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13
0,2025-12-22T19:14:49.000Z,72297342688245,mishamanmai2004@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(1038742538391076),1222-182653-qr7k1m6j-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 7611, numOutputBytes -> 254775)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13
